Rag pipeline

In [85]:
import os
from langchain_community.document_loaders import PyMuPDFLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [86]:
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents) #Stores multiple pdf into one
            print(f"Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("textfile")

Found 1 PDF files to process

Processing: main.pdf
Loaded 17 pages

Total documents loaded: 17


In [87]:
#Chunking
def spli_documents(documents , chunk_sizes = 1000 , chunk_overlaps = 200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_sizes ,
        chunk_overlap = chunk_overlaps,
        separators=["\n\n", "\n", " ", ""]
    )
    split_doc = text_splitter.split_documents(documents)
    print(f"The document length : {len(documents)} splitted into : {len(split_doc)} chunks")

    if split_doc:
        print("Example Chunks")
        print(f"Content: {split_doc[0].page_content[:200]}...")
        print(f"Metadata: {split_doc[0].metadata}")
    return split_doc

In [88]:
chunks = spli_documents(all_pdf_documents)
chunks

The document length : 17 splitted into : 41 chunks
Example Chunks
Content: TRIBHUVAN UNIVERSITY
INSTITUTE OF ENGINEERING
PASHCHIMANCHAL CAMPUS
A Minor Project Report on
Project Title
Submitted By:
Aananda Bastola
[PAS079BCT002]
Abhiujan Baral
[PAS079BCT004]
Bishesh Tiwari
[P...
Metadata: {'producer': 'pdfTeX-1.40.28', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-05-31T16:05:33+00:00', 'source': 'textfile\\main.pdf', 'file_path': 'textfile\\main.pdf', 'total_pages': 17, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-05-31T16:05:33+00:00', 'trapped': '', 'modDate': 'D:20260531160533Z', 'creationDate': 'D:20260531160533Z', 'page': 0, 'source_file': 'main.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.28', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-05-31T16:05:33+00:00', 'source': 'textfile\\main.pdf', 'file_path': 'textfile\\main.pdf', 'total_pages': 17, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-05-31T16:05:33+00:00', 'trapped': '', 'modDate': 'D:20260531160533Z', 'creationDate': 'D:20260531160533Z', 'page': 0, 'source_file': 'main.pdf', 'file_type': 'pdf'}, page_content='TRIBHUVAN UNIVERSITY\nINSTITUTE OF ENGINEERING\nPASHCHIMANCHAL CAMPUS\nA Minor Project Report on\nProject Title\nSubmitted By:\nAananda Bastola\n[PAS079BCT002]\nAbhiujan Baral\n[PAS079BCT004]\nBishesh Tiwari\n[PAS079BCT013]\nSujal Luitel\n[PAS079BCT042]\nSubmitted To:\nDepartment of Electronics and Computer Engineering\nPashchimanchal Campus\nPokhara, Nepal\nJune 1, 2026'),
 Document(metadata={'producer': 'pdfTeX-1.40.28', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-05-31T16:05:33+00:00', 'so

Embedding and Vectordb


In [89]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from sklearn.metrics.pairwise import cosine_similarity
from typing import List , Dict  , Any , Tuple

In [90]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"): #Open source model
        
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"{self.model_name} is loading")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Loaded sucessffully")
        except Exception as e:
            print("Error loading model")
            raise
    
    def generate_embedding(self , texts : List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not Loaded")
        
        print("Letsssssss goooo")
        embeddings = self.model.encode(texts , show_progress_bar=True)
        print("Embedding Completed")
        return embeddings

In [91]:
embedding = EmbeddingManager() #Intialize the embedding manager
embedding

all-MiniLM-L6-v2 is loading
Model Loaded sucessffully


VectorDB


In [92]:
class VectorStore:
    def __init__(self , collection_name : str = "pdf_documents" , persist_directory : str = "vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory , exist_ok=True) #Initializes the directory
            self.client = chromadb.PersistentClient(path = self.persist_directory)  #initializes the chromadb

            self.collection = self.client.get_or_create_collection(
                name  = self.collection_name, #Cha bhaney tehi db use garcha chaina bhaney new create garcha
            )

            print(f"The Database initialized with {self.collection_name}")
        except Exception as e: 
            print("Error initliazing the Database")
            raise
    
    def add_documents(self , document : List[Any] , embeddings : np.ndarray):
        if len(document) != len(embeddings):
            raise ValueError("Number of the document is not equal to the Embbedings")
        
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(document, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(document)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore

The Database initialized with pdf_documents


In [93]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.28', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-05-31T16:05:33+00:00', 'source': 'textfile\\main.pdf', 'file_path': 'textfile\\main.pdf', 'total_pages': 17, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-05-31T16:05:33+00:00', 'trapped': '', 'modDate': 'D:20260531160533Z', 'creationDate': 'D:20260531160533Z', 'page': 0, 'source_file': 'main.pdf', 'file_type': 'pdf'}, page_content='TRIBHUVAN UNIVERSITY\nINSTITUTE OF ENGINEERING\nPASHCHIMANCHAL CAMPUS\nA Minor Project Report on\nProject Title\nSubmitted By:\nAananda Bastola\n[PAS079BCT002]\nAbhiujan Baral\n[PAS079BCT004]\nBishesh Tiwari\n[PAS079BCT013]\nSujal Luitel\n[PAS079BCT042]\nSubmitted To:\nDepartment of Electronics and Computer Engineering\nPashchimanchal Campus\nPokhara, Nepal\nJune 1, 2026'),
 Document(metadata={'producer': 'pdfTeX-1.40.28', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-05-31T16:05:33+00:00', 'so

In [94]:
#Text to embedding
texts = [docs.page_content for docs in chunks]
embeddings = embedding.generate_embedding( texts )
vectorstore.add_documents(chunks , embeddings)


Letsssssss goooo


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding Completed
Successfully added 41 documents to vector store
Total documents in collection: 164


Retreiver Pipeline


In [95]:
class Retreiver:
    def __init__(self , vectorstore : VectorStore , embedding : EmbeddingManager):
        self.vectorstore = vectorstore
        self.embedding = embedding

    def retrive(self , query : str , top_k : int = 5 , similarity_thershodl : float = 0.0) -> List[Dict[str , Any]]:
        print(f"Retreving the query for {query}")
        qurery_embedding = self.embedding.generate_embedding([query])[0]

        try:
            reasults = self.vectorstore.collection.query(
                query_embeddings= [qurery_embedding.toList()],
                n_results= top_k
            )

            retreived_docs = []
            if reasults['documents'] & reasults['documents'][0]:
                document = reasults['documents'][0]
                metadata = reasults['metadatas'][0]
                distance = reasults['distances'][0]
                ids = reasults['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, document, metadata, distance)):
                    similarity_score = 1 - distance

                    if similarity_score >= similarity_thershodl:
                        retreived_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                        print(f"Retrieved {len(retreived_docs)} documents (after filtering)")
                    else:
                        print("No documents found")

                    return retreived_docs
                                    
        except Exception as e:
            print("Error during retreival")
            return []
        
rag_retreiver = Retreiver(vectorstore , embedding)
            
        
    

In [96]:
rag_retreiver.retrive("What is waste water?")

Retreving the query for What is waste water?
Letsssssss goooo


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding Completed
Error during retreival


[]